In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [5]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# Useful constants
root2 = math.sqrt(2)
root3 = math.sqrt(3)

denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 


W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
A2_transform = Operator(W_transform_matrix) 
B1_transform = Operator(W_transform_matrix)


V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
B3_transform = Operator(V_transform_matrix) 

X_transform_matrix = [ [ 1 / root2 , 1 / root2 ],
                      [ 1 / root2, -1 / root2 ] ]
A1_transform = Operator(X_transform_matrix) # this is just the Hadamard gate as an operator 

Z_transform_matrix = [ [ 1 , 0 ],
                      [ 0 , 1 ] ]
A3_transform = Operator(Z_transform_matrix)
B2_transform = Operator(Z_transform_matrix)# this is just the Identity matrix as an operator

rng_transform_matrix = [ [ 1 / root3 , root2 / root3 ],
                           [ root2 / root3 , -1 / root3 ] ]
rng_transform = Operator(rng_transform_matrix) # operator used to create 1/√3 |0⟩ + √2/√3 |1⟩

# Returns 1,2 or 3 with equal probability
def random():
    q = QuantumCircuit(1) 
    q.unitary(rng_transform,[0])
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    #probability of getting 0 is 1/3 so if we measure 0 we can return basis 1
    if counts.get("0",0) == 1:
        return 1
    #probability of getting 1 is 2/3 so we need to do another measurement to determine between basis 2 and 3
    #this uses state 1/√2 |0⟩ + 1/√2 |1⟩ in order to have an equal probability between basis 2 and 3
    else:
        q = QuantumCircuit(1) 
        q.h(0) 
        q.measure_all() 
        backend = BasicSimulator()
        compiled = transpile(q, backend)
        job_sim = backend.run(compiled, shots=1)
        result_sim = job_sim.result() 
        counts2 = result_sim.get_counts(compiled)
        if counts2.get("0",0) == 1:
            return 2
        else:
            return 3

#returns a list of N random bases (where bases are 1,2 or 3)
def random_bases(N):
    bases = []
    for i in range(N):
        bases.append(random())
    return bases

# Construct a circuit that prepares the state  1/sqrt(2)( |01> - |10> )
def entangledPair():
    q = QuantumCircuit(2) 
    q.h(0)
    q.cx(0,1)
    q.z(0)
    q.x(0)
    
    return q

# The length of key trying to be established
N = 50

# this simulates an attack where the attacker makes a measurement 
# constructs a circuit, the attacker then measures the ciruict, applies Alice and Bob's operators, simulate and measure result
# the flag return_bits is used to specify whether we want to return the measured bits or return +/- 1 for entanglement test
def measure_pair_attacker_makes_a_measurement(aliceOp, bobOp, return_bits = False):
    circuit = entangledPair()
    #attacker measures the entangled state
    circuit.measure_all()
    circuit.unitary(aliceOp, [0])
    circuit.unitary(bobOp, [1])
    circuit.measure_all()
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    # get the value that the attacker measured so that we can figure out what Alice and Bob measured
    key = list(counts.keys())[0]
    attacker_measure = key.split()[1]
    count00 = counts.get("00 " + attacker_measure,0) 
    count01 = counts.get("01 " + attacker_measure,0) 
    count10 = counts.get("10 " + attacker_measure,0) 
    count11 = counts.get("11 " + attacker_measure,0)
    # if we want to return Alice and Bob's bits for constructing the key then do so otherwise return 1 or -1 for use in the entanglement test
    if return_bits:
        if count01 == 1:
            return 0,1
        elif count10 == 1:
            return 1,0
        # if we measure 00 or 11 that means Alice and Bob's bits are the same which should not be the case
        elif count00 == 1:
            #print("Bob and Alice measured same bit value, something has gone wrong")
            return 0,0
        elif count11 == 1:
            #print("Bob and Alice measured same bit value, something has gone wrong")
            return 1,1
            
    # The return value includes the conversion from measurement results 0,1 to +1,-1
    # 00 means a value of  1 (+1 * +1)
    # 01 means a value of -1 (+1 * -1)
    # 10 means a value of -1 (-1 * +1)
    # 11 means a value of  1 (+1 * +1)
    else:
        return (count00 - count01 - count10 + count11)


# this simulates an attack where the attacker applies an X gate to Alice's bits
# could also be used to simulate some form of noise
# constructs a circuit, the attacker then applies X to Alice's bit, applies Alice and Bob's operators, simulate and measure result
# the flag return_bits is used to specify whether we want to return the measured bits or return +/- 1 for entanglement test
def measure_pair_attacker_flips_alice_qubit(aliceOp, bobOp, return_bits = False):
    circuit = entangledPair()
    # attacker flips Alice's qubit
    circuit.x(0)
    circuit.unitary(aliceOp, [0])
    circuit.unitary(bobOp, [1])
    circuit.measure_all()
    backend = BasicSimulator()
    compiled = transpile(circuit, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0)
    # if we want to return Alice and Bob's bits for constructing the key then do so otherwise return 1 or -1 for use in the entanglement test
    if return_bits:
        if count01 == 1:
            return 0,1
        elif count10 == 1:
            return 1,0
        # if we measure 00 or 11 that means Alice and Bob's bits are the same which should not be the case
        elif count00 == 1:
            #print("Bob and Alice measured same bit value, something has gone wrong")
            return 0,0
        elif count11 == 1:
            #print("Bob and Alice measured same bit value, something has gone wrong")
            return 1,1
            
    # The return value includes the conversion from measurement results 0,1 to +1,-1
    # 00 means a value of  1 (+1 * +1)
    # 01 means a value of -1 (+1 * -1)
    # 10 means a value of -1 (-1 * +1)
    # 11 means a value of  1 (+1 * +1)
    else:
        return (count00 - count01 - count10 + count11)


# this runs the entire protocol in which the key is generated and the entanglement test value S is calculated
# attack is one of the two functions defined above which simulate a different type of attack
# eg measure_pair_attacker_makes_a_measurement or measure_pair_attacker_flips_alice_qubit
def run(attack):
    # generate bases choices for Alice and Bob
    aliceBases = random_bases(N * 9 // 2)
    bobBases = random_bases(N * 9 // 2)

    # define variables for use in entanglement test calculation/key construction
    A1_B1_total = 0
    A1_B3_total = 0
    A3_B1_total = 0
    A3_B3_total = 0
    A1_B1_count = 0
    A1_B3_count = 0
    A3_B1_count = 0
    A3_B3_count = 0
    alice_key = []
    bob_key = []
    
    # for each pair of basis choices call the attack function which is passed in with the appropriate arguments
    # this will either extend Alice and Bob's keys by 1 bit or continue to collect data for the entanglement test calculation
    for i in range(N * 9 // 2):
        if aliceBases[i] == 1 and bobBases[i] == 1:
            A1_B1_total += attack(A1_transform, B1_transform)
            A1_B1_count+=1
        elif aliceBases[i] == 1 and bobBases[i] == 3:
            A1_B3_total += attack(A1_transform, B3_transform)
            A1_B3_count+=1
        elif aliceBases[i] == 3 and bobBases[i] == 1:
            A3_B1_total += attack(A3_transform, B1_transform)
            A3_B1_count+=1
        elif aliceBases[i] == 3 and bobBases[i] == 3:
            A3_B3_total += attack(A3_transform, B3_transform)
            A3_B3_count+=1
        elif aliceBases[i] == 3 and bobBases[i] == 2:
            alice_bit, bob_bit = attack(A3_transform, B2_transform, True)
            alice_key.append(alice_bit)
            # Invert Bob's bit
            if bob_bit == 1:
                bob_key.append(0)
            else:
                bob_key.append(1)
        elif aliceBases[i] == 2 and bobBases[i] == 1:
            alice_bit, bob_bit = attack(A2_transform, B1_transform, True)
            alice_key.append(alice_bit)
            # Invert Bob's bit
            if bob_bit == 1:
                bob_key.append(0)
            else:
                bob_key.append(1)
    
    # The final calculuation
    # ensure we have enough data and also avoid division by zero errors
    if 0 in [A1_B1_count, A1_B3_count,A3_B1_count,A3_B3_count]:
        print("Not enough data to calulate S")
    else:    
        result = abs(A1_B1_total/A1_B1_count - A1_B3_total/A1_B3_count + A3_B1_total/A3_B1_count + A3_B3_total/A3_B3_count)
    
        print("S =",result)
        print("2√2 =",2*root2)
        print("Alice key:",alice_key)
        print("Bob key:",bob_key)
        print("Length of Alice key:",len(alice_key))
        print("Alice and Bob have same key:",alice_key == bob_key)
        if result <= 2:
            print("Bell inequality violated, definite interference detected")
    
# runs both simulations
print("Measurement attack")
run(measure_pair_attacker_makes_a_measurement)
print()
print("Alice bit flip attack")
run(measure_pair_attacker_flips_alice_qubit)
print()

# I've showed the results from two different types of meaningful attacks, one passive (measurement) and one active (bit flip).
# There are other possible attacks which could be detected in similar ways.

# The first of these is the measurement attack. This is simply when the attacker intercepts the qubits and 
# performs a measurement on them. Measuring the qubits breaks the entangled state of the qubits hence breaking the protocol.
# Due to the No Cloning Theorem, the attacker can't capture the state of a qubit and still resend the same qubits back
# to Alice and Bob. This means that parts of the key will not agree which we can detect.
# Since the entanglement is broken we will find that Bell's inequality is violated.

# The second attack is a bit flip attack. This simulates an attacker flipping the state of Alice's bit.
# While this arguably isnt as much of an issue as the measurement attack as the attacker is gaining no knowledge
# regarding the states of the qubits, it is still a valid attack that a malicious user could perform
# in attempt to disrupt the protocol and prevent Alice and Bob from communicating securly.
# Flipping Alice's bit changes the state into a different, undesired, Bell state.

# In these examples you can see that we can actually detect the attacks in two major ways.
# Firstly we can see that the value of S is significantly below 2√2 and hence an attack has occured as Bell's inequality is violated.
# The attack can also be detected if Alice and Bob decided to compare a section of their key where
# they would find out, with high probability (higher the more bits they compare), that their keys 
# are not the same as eachother and therefore something has gone wrong, eg an attack.

Measurement attack
S = 1.6072037796175727
2√2 = 2.8284271247461903
Alice key: [1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]
Bob key: [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1]
Length of Alice key: 48
Alice and Bob have same key: False
Bell inequality violated, definite interference detected

Alice bit flip attack
S = 0.3734042069874154
2√2 = 2.8284271247461903
Alice key: [1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0]
Bob key: [0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]
Length of Alice key: 45
Alice and Bob have same key: False
Bell inequality violated, definite interference detected

